In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [2]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|       grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------+--------------------+------+-----+
|69d66fbc2230a451e...|  Mystery|2026-04-08 15:09:47|G1_Ecommerce|It's Only the Him...|     1|45.17|
|69d66fbc2230a451e...|  Mystery|2026-04-08 15:09:47|G1_Ecommerce|Full Moon over No...|     1|49.43|
|69d66fbc2230a451e...|  Mystery|2026-04-08 15:09:47|G1_Ecommerce|See America: A Ce...|     1|48.87|
|69d66fbc2230a451e...|  Mystery|2026-04-08 15:09:48|G1_Ecommerce|Vagabonding: An U...|     1|36.94|
|69d66fbc2230a451e...|  

In [3]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 22
Registros validos: 22


In [4]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|It's Only the Him...|45.17|
|Full Moon over No...|49.43|
|See America: A Ce...|48.87|
|Vagabonding: An U...|36.94|
|Under the Tuscan Sun|37.33|
|  A Summer In Europe|44.34|
|The Great Railway...|30.54|
|A Year in Provenc...|56.88|
|The Road to Littl...|23.21|
|Neither Here nor ...|38.95|
|1,000 Places to S...|26.08|
|It's Only the Him...|45.17|
|Full Moon over No...|49.43|
|See America: A Ce...|48.87|
|Vagabonding: An U...|36.94|
|Under the Tuscan Sun|37.33|
|  A Summer In Europe|44.34|
|The Great Railway...|30.54|
|A Year in Provenc...|56.88|
|The Road to Littl...|23.21|
+--------------------+-----+
only showing top 20 rows



In [7]:
df.filter(df["valor"] > 50).show()

+--------------------+---------+-------------------+------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|       grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------+--------------------+------+-----+
|69d66fbc2230a451e...|  Mystery|2026-04-08 15:09:48|G1_Ecommerce|A Year in Provenc...|     1|56.88|
|69d66fbf2230a451e...|  Mystery|2026-04-08 15:09:51|G1_Ecommerce|A Year in Provenc...|     1|56.88|
+--------------------+---------+-------------------+------------+--------------------+------+-----+



In [8]:
df.groupBy("grupo").count().show()

+------------+-----+
|       grupo|count|
+------------+-----+
|G1_Ecommerce|   22|
+------------+-----+



In [9]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+--------------+---------------+------------------+-------------+-------------+
|     categoria|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+--------------+---------------+------------------+-------------+-------------+
|   Young Adult|             54|35.449074074074076|         10.0|        58.14|
|Sequential Art|             60| 33.84350000000001|        10.16|        57.62|
+--------------+---------------+------------------+-------------+-------------+



In [10]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Aristotle and Dante Discover the Secrets of the Universe (Aristotle and Dante Discover the Secrets of the Universe #1)
Categoría: Young Adult
Fecha y hora de captura: 2026-04-08 15:51:49
Valor: 58.14
